In [129]:
import json
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("S3_to_Iceberg_Bronze")
    .master("local[*]")
    # Added iceberg-aws-bundle to provide the Glue client dependencies
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.3,org.apache.iceberg:iceberg-aws-bundle:1.4.3")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    
    # Configure Iceberg to use AWS Glue Data Catalog
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://us-recipies-bronze-v1/iceberg-warehouse/")
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    
    # AWS S3 Timeout overrides
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000")
    .config("spark.hadoop.fs.s3a.socket.timeout", "60000")
    .getOrCreate()
)

# Route Spark's internal Hadoop connection to AWS S3
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.profile.ProfileCredentialsProvider")
hadoop_conf.set("fs.s3a.endpoint", "s3.amazonaws.com")

In [2]:

# Read the messy JSON from S3
s3_path = "s3a://us-recipies-bronze-v1/bronze/US_recipes.json"


In [3]:
# 1. Access the underlying Java classes via PySpark's JVM gateway
URI = spark.sparkContext._gateway.jvm.java.net.URI
Path = spark.sparkContext._gateway.jvm.org.apache.hadoop.fs.Path
FileSystem = spark.sparkContext._gateway.jvm.org.apache.hadoop.fs.FileSystem

# 2. Define your S3 path
s3_path_str = "s3a://us-recipies-bronze-v1/bronze/US_recipes.json"
path = Path(s3_path_str)

# 3. Initialize the Hadoop FileSystem object with your S3 configuration
fs = FileSystem.get(URI(s3_path_str), spark.sparkContext._jsc.hadoopConfiguration())

# 4. Check existence and print metadata
if fs.exists(path):
    print("✅ Connection successful: File exists in S3!")
    
    # Check the file size to understand why the heap is crashing
    file_length_bytes = fs.getFileStatus(path).getLen()
    print(f"File Size: {file_length_bytes / (1024 * 1024):.2f} MB")
else:
    print("❌ Connection successful, but file was not found. Check the path.")

✅ Connection successful: File exists in S3!
File Size: 13.16 MB


In [20]:
from pyspark import SparkContext
import json


rdd = spark.sparkContext.wholeTextFiles(s3_path)

In [21]:
raw_data = rdd.values().first() # get the first file from the folder

In [22]:
json_data = json.loads(raw_data) #load it as json
parsed_data = list(json_data.values())# now parse the data by value
# lets get the output in list 
list_dump = [json.dumps(record) for record in parsed_data] 

#lets parallaize this now
df_par = spark.sparkContext.parallelize(list_dump)

# remeber its loads, dumps not load & dump

In [23]:
df = spark.read.json(df_par)

In [31]:
df.show(5)

+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------------------+---------+------+----------+--------------------+----------+
|     Contient|Country_State|                 URL|cook_time|         cuisine|         description|         ingredients|        instructions|           nutrients|prep_time|rating|    serves|               title|total_time|
+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------------------+---------+------+----------+--------------------+----------+
|North America|           US|https://www.allre...|    100.0|Southern Recipes|Shared from a Sou...|[1 (1 pound) swee...|[Place whole swee...|{"calories":"389 ...|     15.0|   4.8|8 servings|    Sweet Potato Pie|     115.0|
|North America|           US|https://www.allre...|     40.0|Southern Recipes|This peach cobble...|[8 fresh peach

In [4]:
spark.sql("USE glue_catalog")

DataFrame[]

In [37]:
# 1. Create the database namespace in AWS Glue
spark.sql("CREATE DATABASE IF NOT EXISTS glue_catalog.bronze_recipes")

# 2. Define the target table
table_id = "glue_catalog.bronze_recipes.us_raw"

df.writeTo(table_id).using('iceberg').createOrReplace()

In [29]:
spark.sql('SHOW DATABASES IN glue_catalog').show(truncate=False)

+--------------+
|namespace     |
+--------------+
|bronze_recipes|
|default       |
+--------------+



In [38]:
spark.sql('SHOW TABLES IN glue_catalog.bronze_recipes').show()

+--------------+---------+-----------+
|     namespace|tableName|isTemporary|
+--------------+---------+-----------+
|bronze_recipes|   us_raw|      false|
+--------------+---------+-----------+



In [47]:
spark.sql('SELECT * FROM glue_catalog.bronze_recipes.us_raw').toPandas()

,Contient,Country_State,URL,cook_time,cuisine,description,ingredients,instructions,nutrients,prep_time,rating,serves,title,total_time
0,North America,US,https://www.allrecipes.com/recipe/12142/sweet-...,100.0,Southern Recipes,"Shared from a Southern recipe, this homemade s...","[1 (1 pound) sweet potato, with skin, 0.5 cup ...",[Place whole sweet potato in pot and cover wit...,"{""calories"":""389 kcal"",""carbohydrateContent"":""...",15.0,4.8,8 servings,Sweet Potato Pie,115.0
1,North America,US,https://www.allrecipes.com/recipe/51535/fresh-...,40.0,Southern Recipes,This peach cobbler recipe makes the perfect de...,"[8 fresh peaches - peeled, pitted and sliced i...",[Preheat oven to 425 degrees F (220 degrees C)...,"{""calories"":""562 kcal"",""carbohydrateContent"":""...",20.0,4.7,4 servings,Fresh Southern Peach Cobbler,60.0
2,North America,US,https://www.allrecipes.com/recipe/16760/best-f...,15.0,Southern Recipes,Fried green tomatoes are a quick and easy way ...,"[4 large green tomatoes, 2 eggs, 0.5 cup milk,...",[Slice tomatoes 1/2 inch thick. Discard the en...,"{""calories"":""510 kcal"",""carbohydrateContent"":""...",5.0,4.7,4 servings,Best Fried Green Tomatoes,20.0
3,North America,US,https://www.allrecipes.com/recipe/31848/jambal...,45.0,Southern Recipes,"A spicy jambalaya with chicken, andouille saus...","[2 tablespoons peanut oil, divided, 1 tablespo...","[Gather all ingredients., Heat 1 tablespoon of...","{""calories"":""465 kcal"",""carbohydrateContent"":""...",20.0,4.8,6 servings,Best Jambalaya,65.0
4,North America,US,https://www.allrecipes.com/recipe/58211/authen...,185.0,Southern Recipes,This easy authentic Louisiana red beans and ri...,"[1 pound dry kidney beans, 0.25 cup olive oil,...","[Gather all ingredients., Rinse beans, and the...","{""calories"":""630 kcal"",""carbohydrateContent"":""...",25.0,4.5,8 servings,Authentic Louisiana Red Beans and Rice,690.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8446,North America,US,https://www.allrecipes.com/recipe/278333/amish...,15.0,Amish and Mennonite Recipes,This Amish grandmother's chocolate pudding rec...,"[2 cups milk, or more as needed, 2 cups white ...","[Combine milk, sugar, cocoa, and flour in a sa...","{""calories"":""274 kcal"",""carbohydrateContent"":""...",5.0,3.3,8 servings,Amish Chocolate Pudding,20.0
8447,North America,US,https://www.allrecipes.com/recipe/139695/geras...,30.0,Amish and Mennonite Recipes,Amish cooking is amazingly simple and incredib...,"[1 cup white sugar, 0.5 cup cocoa powder, 0.75...",[Preheat oven to 375 degrees F (190 degrees C)...,"{""calories"":""343 kcal"",""carbohydrateContent"":""...",25.0,3.6,16 servings,Gera's Amish Funny Cake,55.0
8448,North America,US,https://www.allrecipes.com/recipe/232881/quick...,30.0,Amish and Mennonite Recipes,This Amish banana nut cake is a delightful sna...,"[1 cup white sugar, 0.5 cup butter, 0.25 cup m...",[Preheat oven to 375 degrees F (190 degrees C)...,"{""calories"":""273 kcal"",""carbohydrateContent"":""...",15.0,3.0,12 servings,Quick Banana Nut Cake,45.0
8449,North America,US,https://www.allrecipes.com/recipe/278337/amish...,45.0,Amish and Mennonite Recipes,It doesn't get easier or more delicious than t...,"[1 cup sour cream, 1 egg, 0.75 cup white sugar...",[Preheat the oven to 400 degrees F (200 degree...,"{""calories"":""409 kcal"",""carbohydrateContent"":""...",15.0,NaN,8 servings,Amish Sour Cream Apple Pie,60.0


In [8]:
spark.sql('SELECT * FROM glue_catalog.bronze_recipes.us_raw.history').show()

+--------------------+-------------------+---------+-------------------+
|     made_current_at|        snapshot_id|parent_id|is_current_ancestor|
+--------------------+-------------------+---------+-------------------+
|2026-11-04 17:48:...|2378449303176395676|     NULL|               true|
+--------------------+-------------------+---------+-------------------+



In [38]:
df = spark.sql('SELECT * FROM glue_catalog.bronze_recipes.us_raw')

In [11]:
df.show(5)

+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------------------+---------+------+----------+--------------------+----------+
|     Contient|Country_State|                 URL|cook_time|         cuisine|         description|         ingredients|        instructions|           nutrients|prep_time|rating|    serves|               title|total_time|
+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------------------+---------+------+----------+--------------------+----------+
|North America|           US|https://www.allre...|    100.0|Southern Recipes|Shared from a Sou...|[1 (1 pound) swee...|[Place whole swee...|{"calories":"389 ...|     15.0|   4.8|8 servings|    Sweet Potato Pie|     115.0|
|North America|           US|https://www.allre...|     40.0|Southern Recipes|This peach cobble...|[8 fresh peach

In [12]:
df.printSchema()

root
 |-- Contient: string (nullable = true)
 |-- Country_State: string (nullable = true)
 |-- URL: string (nullable = true)
 |-- cook_time: double (nullable = true)
 |-- cuisine: string (nullable = true)
 |-- description: string (nullable = true)
 |-- ingredients: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- instructions: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- nutrients: string (nullable = true)
 |-- prep_time: double (nullable = true)
 |-- rating: double (nullable = true)
 |-- serves: string (nullable = true)
 |-- title: string (nullable = true)
 |-- total_time: double (nullable = true)



In [37]:
spark.sql('DESCRIBE glue_catalog.bronze_recipes.us_raw').show(truncate=False)

+-------------+-------------+-------+
|col_name     |data_type    |comment|
+-------------+-------------+-------+
|Contient     |string       |NULL   |
|Country_State|string       |NULL   |
|URL          |string       |NULL   |
|cook_time    |double       |NULL   |
|cuisine      |string       |NULL   |
|description  |string       |NULL   |
|ingredients  |array<string>|NULL   |
|instructions |array<string>|NULL   |
|nutrients    |string       |NULL   |
|prep_time    |double       |NULL   |
|rating       |double       |NULL   |
|serves       |string       |NULL   |
|title        |string       |NULL   |
|total_time   |double       |NULL   |
+-------------+-------------+-------+



In [46]:
cols = [f"f.col('{col}')," for col in df.columns]
print("\n".join(cols))

f.col('Contient'),
f.col('Country_State'),
f.col('URL'),
f.col('cook_time'),
f.col('cuisine'),
f.col('description'),
f.col('ingredients'),
f.col('instructions'),
f.col('nutrients'),
f.col('prep_time'),
f.col('rating'),
f.col('serves'),
f.col('title'),
f.col('total_time'),


In [50]:
from pyspark.sql import functions as f

df_silver_relational = (
    df
    .withColumn('nutrients_exploded',f.explode(f.col('nutrients')))
    # First, bring all nested struct fields to the top level
    .select(
       f.col('Contient'),
    f.col('Country_State'),
    f.col('URL'),
    f.col('cook_time'),
    f.col('cuisine'),
    f.col('description'),
    f.col('ingredients'),
    f.explode(f.col('instructions')),
  #  f.explode(f.col('nutrients_exploded')).alias('nutrients_exploded'),
    f.col('prep_time'),
    f.col('rating'),
    f.col('serves'),
    f.col('title'),
    f.col('total_time')
    )
    # Next, explode the array so each ingredient gets its own row
    .withColumn("ingredient", explode(col("ingredients")))
    # Finally, drop the original array column to keep the table strictly relational
    .drop("ingredients")
)

AnalysisException: [DATATYPE_MISMATCH.UNEXPECTED_INPUT_TYPE] Cannot resolve "explode(nutrients)" due to data type mismatch: Parameter 1 requires the ("ARRAY" or "MAP") type, however "nutrients" has the type "STRING".;
'Project [Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329, explode(nutrients#324) AS nutrients_exploded#359]
+- Project [Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329]
   +- SubqueryAlias glue_catalog.bronze_recipes.us_raw
      +- RelationV2[Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329] glue_catalog.bronze_recipe

In [60]:
from pyspark.sql.types import StructType

df.withColumn('nutrient_struct',f.explode(f.col('nutrients').cast(StructType())))

AnalysisException: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "nutrients" due to data type mismatch: cannot cast "STRING" to "STRUCT<>".;
'Project [Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329, explode(cast(nutrients#324 as struct<>)) AS nutrient_struct#362]
+- Project [Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329]
   +- SubqueryAlias glue_catalog.bronze_recipes.us_raw
      +- RelationV2[Contient#316, Country_State#317, URL#318, cook_time#319, cuisine#320, description#321, ingredients#322, instructions#323, nutrients#324, prep_time#325, rating#326, serves#327, title#328, total_time#329] glue_catalog.bronze_recipes.us_raw glue_catalog.bronze_recipes.us_raw


SyntaxError: unterminated string literal (detected at line 1) (2252294067.py, line 1)

In [66]:
#spark is not able to parse the nested struct so better option is to define the schema while creating iceberg table
#lets read the column names first then

#lets fetch single non null sample string
sample_json = df.filter(
    f.col('nutrients').isNotNull()
).select('nutrients').first()[0]



In [79]:
import json 

data_dict = json.loads(sample_json)
data_dict.keys()

dict_keys(['calories', 'carbohydrateContent', 'cholesterolContent', 'fiberContent', 'proteinContent', 'saturatedFatContent', 'sodiumContent', 'sugarContent', 'fatContent', 'unsaturatedFatContent'])

In [85]:
#get the schem of json
schema_dll = spark.sql(f"select schema_of_json('{sample_json}')").first()[0]

In [86]:
print(schema_dll)

STRUCT<calories: STRING, carbohydrateContent: STRING, cholesterolContent: STRING, fatContent: STRING, fiberContent: STRING, proteinContent: STRING, saturatedFatContent: STRING, sodiumContent: STRING, sugarContent: STRING, unsaturatedFatContent: STRING>


In [90]:
spark.sql(
    """
        ALTER TABLE glue_catalog.bronze_recipes.us_raw
        ALTER COLUMN nutrients TYPE STRUCT<calories: STRING, carbohydrateContent: STRING, cholesterolContent: STRING, fatContent: STRING, fiberContent: STRING, proteinContent: STRING, saturatedFatContent: STRING, sodiumContent: STRING, sugarContent: STRING, unsaturatedFatContent: STRING>
    """
)

"""
Iceberg's ALTER COLUMN ... TYPE command strictly supports widening primitive types (such as changing an INT to a BIGINT). It physically cannot convert an existing STRING column into a complex nested STRUCT because the underlying Parquet files have already saved the historical data as flat text.
"""

AnalysisException: [CANNOT_UPDATE_FIELD.STRUCT_TYPE] Cannot update `glue_catalog`.`bronze_recipes`.`us_raw` field `nutrients` type: Update a struct by updating its fields.; line 2 pos 8;
AlterColumn resolvedfieldname(StructField(nutrients,StringType,true)), StructType(StructField(calories,StringType,true),StructField(carbohydrateContent,StringType,true),StructField(cholesterolContent,StringType,true),StructField(fatContent,StringType,true),StructField(fiberContent,StringType,true),StructField(proteinContent,StringType,true),StructField(saturatedFatContent,StringType,true),StructField(sodiumContent,StringType,true),StructField(sugarContent,StringType,true),StructField(unsaturatedFatContent,StringType,true))
+- ResolvedTable org.apache.iceberg.spark.SparkCatalog@33e84ce9, bronze_recipes.us_raw, glue_catalog.bronze_recipes.us_raw, [Contient#415, Country_State#416, URL#417, cook_time#418, cuisine#419, description#420, ingredients#421, instructions#422, nutrients#423, prep_time#424, rating#

In [91]:
# lets rename the column n create a new one in this case then

spark.sql("""
    ALTER TABLE glue_catalog.bronze_recipes.us_raw
    RENAME COLUMN nutrients TO nutrients_legacy_string
    """
)

DataFrame[]

In [93]:
spark.sql('DESCRIBE glue_catalog.bronze_recipes.us_raw').show()

+--------------------+-------------+-------+
|            col_name|    data_type|comment|
+--------------------+-------------+-------+
|            Contient|       string|   NULL|
|       Country_State|       string|   NULL|
|                 URL|       string|   NULL|
|           cook_time|       double|   NULL|
|             cuisine|       string|   NULL|
|         description|       string|   NULL|
|         ingredients|array<string>|   NULL|
|        instructions|array<string>|   NULL|
|nutrients_legacy_...|       string|   NULL|
|           prep_time|       double|   NULL|
|              rating|       double|   NULL|
|              serves|       string|   NULL|
|               title|       string|   NULL|
|          total_time|       double|   NULL|
+--------------------+-------------+-------+



In [95]:
spark.sql(
    """
        ALTER TABLE glue_catalog.bronze_recipes.us_raw
        ADD COLUMNS (nutrients STRUCT<
            calories: STRING, carbohydrateContent: STRING,
            cholesterolContent: STRING, fatContent: STRING,
            fiberContent: STRING, proteinContent: STRING,
            saturatedFatContent: STRING, sodiumContent: STRING,
            sugarContent: STRING, unsaturatedFatContent: STRING
            >
            )
    """
)

DataFrame[]

In [96]:
spark.sql('DESCRIBE glue_catalog.bronze_recipes.us_raw').show()

+--------------------+--------------------+-------+
|            col_name|           data_type|comment|
+--------------------+--------------------+-------+
|            Contient|              string|   NULL|
|       Country_State|              string|   NULL|
|                 URL|              string|   NULL|
|           cook_time|              double|   NULL|
|             cuisine|              string|   NULL|
|         description|              string|   NULL|
|         ingredients|       array<string>|   NULL|
|        instructions|       array<string>|   NULL|
|nutrients_legacy_...|              string|   NULL|
|           prep_time|              double|   NULL|
|              rating|              double|   NULL|
|              serves|              string|   NULL|
|               title|              string|   NULL|
|          total_time|              double|   NULL|
|           nutrients|struct<calories:s...|   NULL|
+--------------------+--------------------+-------+



In [98]:
spark.sql("""
    UPDATE glue_catalog.bronze_recipes.us_raw 
    SET nutrients = from_json(
        nutrients_legacy_string, 
        'calories STRING, carbohydrateContent STRING, cholesterolContent STRING, fatContent STRING, fiberContent STRING, proteinContent STRING, saturatedFatContent STRING, sodiumContent STRING, sugarContent STRING, unsaturatedFatContent STRING'
    )
""")

DataFrame[]

In [99]:
df = spark.sql('SELECT * FROM glue_catalog.bronze_recipes.us_raw')

In [111]:
from pyspark.sql.functions import col

df_unpacked = (
    df
    .select(
      f.col('Contient'),
f.col('Country_State'),
f.col('URL'),
f.col('cook_time'),
f.col('cuisine'),
f.col('description'),
f.col('ingredients'),
f.col('instructions'),
f.col('nutrients.*'),
f.col('prep_time'),
f.col('rating'),
f.col('serves'),
f.col('title'),
f.col('total_time')
    )
)



In [112]:
df_unpacked.limit(5).toPandas()

,Contient,Country_State,URL,cook_time,cuisine,description,ingredients,instructions,calories,carbohydrateContent,...,proteinContent,saturatedFatContent,sodiumContent,sugarContent,unsaturatedFatContent,prep_time,rating,serves,title,total_time
0,North America,US,https://www.allrecipes.com/recipe/12142/sweet-...,100.0,Southern Recipes,"Shared from a Southern recipe, this homemade s...","[1 (1 pound) sweet potato, with skin, 0.5 cup ...",[Place whole sweet potato in pot and cover wit...,389 kcal,48 g,...,5 g,10 g,254 mg,28 g,0 g,15.0,4.8,8 servings,Sweet Potato Pie,115.0
1,North America,US,https://www.allrecipes.com/recipe/51535/fresh-...,40.0,Southern Recipes,This peach cobbler recipe makes the perfect de...,"[8 fresh peaches - peeled, pitted and sliced i...",[Preheat oven to 425 degrees F (220 degrees C)...,562 kcal,99 g,...,4 g,11 g,400 mg,73 g,0 g,20.0,4.7,4 servings,Fresh Southern Peach Cobbler,60.0
2,North America,US,https://www.allrecipes.com/recipe/16760/best-f...,15.0,Southern Recipes,Fried green tomatoes are a quick and easy way ...,"[4 large green tomatoes, 2 eggs, 0.5 cup milk,...",[Slice tomatoes 1/2 inch thick. Discard the en...,510 kcal,56 g,...,13 g,4 g,1136 mg,10 g,0 g,5.0,4.7,4 servings,Best Fried Green Tomatoes,20.0
3,North America,US,https://www.allrecipes.com/recipe/31848/jambal...,45.0,Southern Recipes,"A spicy jambalaya with chicken, andouille saus...","[2 tablespoons peanut oil, divided, 1 tablespo...","[Gather all ingredients., Heat 1 tablespoon of...",465 kcal,42 g,...,28 g,6 g,1633 mg,2 g,0 g,20.0,4.8,6 servings,Best Jambalaya,65.0
4,North America,US,https://www.allrecipes.com/recipe/58211/authen...,185.0,Southern Recipes,This easy authentic Louisiana red beans and ri...,"[1 pound dry kidney beans, 0.25 cup olive oil,...","[Gather all ingredients., Rinse beans, and the...",630 kcal,79 g,...,24 g,7 g,604 mg,3 g,0 g,25.0,4.5,8 servings,Authentic Louisiana Red Beans and Rice,690.0


In [124]:
# To enable Dynamic Partition Overwrite
spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")

(df_unpacked.writeTo("glue_catalog.bronze_recipes.recipes_unpacked_silver")
    .using("iceberg")
    .tableProperty("write.update.mode", "merge-on-read")
    .tableProperty("write.distribution-mode", "hash")
    .tableProperty("write.target-file-size-bytes", "268435456")
    .partitionedBy("Contient")
     .createOrReplace()# <-- Sets up partition on creation
)

In [126]:
spark.sql('SHOW TABLES IN glue_catalog.bronze_recipes').show()

+--------------+--------------------+-----------+
|     namespace|           tableName|isTemporary|
+--------------+--------------------+-----------+
|bronze_recipes|recipes_unpacked_...|      false|
|bronze_recipes|              us_raw|      false|
+--------------+--------------------+-----------+



In [130]:
spark.sql('SELECT * FROM glue_catalog.bronze_recipes.recipes_unpacked_silver').show(5)

+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------+-------------------+------------------+----------+------------+--------------+-------------------+-------------+------------+---------------------+---------+------+----------+--------------------+----------+
|     Contient|Country_State|                 URL|cook_time|         cuisine|         description|         ingredients|        instructions|calories|carbohydrateContent|cholesterolContent|fatContent|fiberContent|proteinContent|saturatedFatContent|sodiumContent|sugarContent|unsaturatedFatContent|prep_time|rating|    serves|               title|total_time|
+-------------+-------------+--------------------+---------+----------------+--------------------+--------------------+--------------------+--------+-------------------+------------------+----------+------------+--------------+-------------------+-------------+------------+------------